In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
# debug add_interaction and dm_idx stuff

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    stepsize_s=0.1,
)
encoder.verify()

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    stepsize_s=0.1,
    strategy_filter="mb",
)
encoder_mb.verify()

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    stepsize_s=0.1,
    strategy_filter="mf",
)
encoder_mf.verify()

## diff heatmaps

In [ ]:
bweight_diff = encoder_mb.encoder_weights - encoder_mf.encoder_weights
bweight_diff = bweight_diff[:, 5:]

fig, axes = plt.subplots(ncols=2, figsize=(6, 2))

axes[0].imshow(
    bweight_diff[encoder.reg_idxs["DMS"]], cmap="coolwarm", vmax=0.5, vmin=-0.5
)
axes[1].imshow(
    bweight_diff[encoder.reg_idxs["DLS"]], cmap="coolwarm", vmax=0.5, vmin=-0.5
)


axes[0].set_title("DMS", loc="left")
axes[1].set_title("DLS", loc="left")

In [ ]:
# HYPOTHESIS: neurons that are different between mb and mf for one regressor tend to be different for other regressors as well
# plot correlation of differences for one regressor against others
import numpy as np
from scipy.stats import pearsonr as r

diff_corr = np.array(
    [
        [
            r(
                bweight_diff[:, encoder.dm_idxs[regr_a] - encoder.num_tents],
                bweight_diff[:, encoder.dm_idxs[regr_b] - encoder.num_tents],
            ).statistic
            for regr_b in encoder.dm_idxs.keys()
            if "tent" not in regr_b
        ]
        for regr_a in encoder.dm_idxs.keys()
        if "tent" not in regr_a
    ]
)

plt.figure()
plt.imshow(diff_corr, vmin=-1, vmax=1, cmap="coolwarm")
plt.xticks(
    ticks=range(len(diff_corr)), labels=list(encoder.dm_idxs.keys())[5:], rotation=90
)
plt.yticks(
    ticks=range(len(diff_corr)), labels=list(encoder.dm_idxs.keys())[5:], rotation=0
)
plt.colorbar()
plt.show()

In [ ]:
diff_corr = np.array(
    [
        [
            r(
                bweight_diff[:, encoder.dm_idxs[regr_a] - encoder.num_tents],
                bweight_diff[:, encoder.dm_idxs[regr_b] - encoder.num_tents],
            ).statistic
            for regr_b in encoder.dm_idxs.keys()
            if "tent" not in regr_b
        ]
        for regr_a in encoder.dm_idxs.keys()
        if "tent" not in regr_a
    ]
)
diff_corr_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff[
                        encoder.reg_idxs[reg],
                        encoder.dm_idxs[regr_a] - encoder.num_tents,
                    ],
                    bweight_diff[
                        encoder.reg_idxs[reg],
                        encoder.dm_idxs[regr_b] - encoder.num_tents,
                    ],
                ).statistic
                for regr_b in encoder.dm_idxs.keys()
                if "tent" not in regr_b
            ]
            for regr_a in encoder.dm_idxs.keys()
            if "tent" not in regr_a
        ]
    )
    for reg in encoder.regions
}


def plot_corr(diff_corr, reg=None):
    if reg is not None:
        diff_corr = diff_corr[reg]
    fig, axes = plt.subplots(
        nrows=4,
        ncols=4,
        figsize=(5, 5),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )

    for i, regr_a in enumerate(encoder.tv_keys):
        for j, regr_b in enumerate(encoder.tv_keys):
            ax = axes[i][j]

            idx_0a, idx_m1a = (
                encoder.dm_idxs[f"{regr_a}_0"] - encoder.num_tents,
                encoder.dm_idxs[f"{regr_a}_{encoder.num_bins - 1}"] - encoder.num_tents,
            )
            idx_0b, idx_m1b = (
                encoder.dm_idxs[f"{regr_b}_0"] - encoder.num_tents,
                encoder.dm_idxs[f"{regr_b}_{encoder.num_bins - 1}"] - encoder.num_tents,
            )

            ax.imshow(
                diff_corr[idx_0a : idx_m1a + 1, idx_0b : idx_m1b + 1],
                vmin=-1,
                vmax=1,
                cmap="coolwarm",
            )

            if j == 0:
                ax.set_ylabel(regr_a)
            if i == len(encoder.tv_keys) - 1:
                ax.set_xlabel(regr_b)
    title = (
        rf"$r$ of bweight difference between strategies, {reg}"
        if reg is not None
        else r"$r$ of bweight difference between strategies"
    )
    fig.suptitle(title)


plot_corr(diff_corr)
plot_corr(diff_corr_reg, reg="DLS")
plot_corr(diff_corr_reg, reg="DMS")

### control

In [ ]:
from core.data import get_strategy_filter_idxs

stepsize_s = 0.1


def build_encoder_strategy_ctrl(strategy="mb", stepsize_s=0.1):
    encoder = Encoder(
        subj_id,
        sess_id,
    )
    encoder.get_data()

    # get an even percentage of model-based and model-free
    idxs = get_strategy_filter_idxs(
        encoder.trial_data, balance_strategy=True, cond_balance=False
    )
    idxs_mb = np.random.choice(idxs["mb"], len(idxs["mb"]) // 2)
    idxs_mf = np.random.choice(idxs["mf"], len(idxs["mf"]) // 2)
    idxs = np.sort(np.concatenate((idxs_mb, idxs_mf)))

    encoder_strategy = make_tre(StrategyEncoder)(
        subj_id,
        sess_id,
        stepsize_s=stepsize_s,
        idxs=idxs,
        strategy_filter=strategy,
    )
    encoder_strategy.fit_encoder()

    assert len(np.unique(encoder_strategy.trial_data["strategy"])) == 2
    assert np.isclose(
        (encoder_strategy.trial_data["strategy"] == 1).mean(), 0.5, atol=0.15
    ), (encoder_strategy.trial_data["strategy"] == 1).mean()

    return encoder_strategy


encoder_mb = build_encoder_strategy_ctrl(strategy="mb", stepsize_s=0.1)
encoder_mf = build_encoder_strategy_ctrl(strategy="mf", stepsize_s=0.1)

In [ ]:
bweight_diff_ctrl = encoder_mb.encoder_weights - encoder_mf.encoder_weights
bweight_diff_ctrl = bweight_diff_ctrl[:, 5:]

fig, axes = plt.subplots(ncols=2, figsize=(6, 2))

axes[0].imshow(
    bweight_diff_ctrl[encoder.reg_idxs["DMS"]], cmap="coolwarm", vmax=0.5, vmin=-0.5
)
axes[1].imshow(
    bweight_diff_ctrl[encoder.reg_idxs["DLS"]], cmap="coolwarm", vmax=0.5, vmin=-0.5
)

axes[0].set_title("DMS", loc="left")
axes[1].set_title("DLS", loc="left")

In [ ]:
from core.viz import plot_scatter, plot_kdes

fig, axes = plt.subplots(nrows=len(encoder.tv_keys), ncols=2, figsize=(5, 3))

for i, reg in enumerate(encoder.regions):
    for j, regr in enumerate(encoder.tv_keys):
        idx_0, idx_m1 = (
            encoder.dm_idxs[f"{regr}_0"],
            encoder.dm_idxs[f"{regr}_{encoder.num_tents - 1}"],
        )

        bweights = {
            "empirical": np.ravel(
                bweight_diff[encoder.reg_idxs[reg], idx_0 : idx_m1 + 1]
            ),
            "control": np.ravel(
                bweight_diff_ctrl[encoder.reg_idxs[reg], idx_0 : idx_m1 + 1]
            ),
        }

        plot_kdes(
            bweights,
            legend=False,
            line_kwargs={
                "empirical": {"color": "#DD2299"},
                "control": {"color": "#555555"},
            },
            ax=axes[j][i],
        )
        # plot_scatter(bweights['control'], bweights['empirical'], ax=axes[j][i])

In [ ]:
diff_corr_ctrl = np.array(
    [
        [
            r(
                bweight_diff_ctrl[:, encoder.dm_idxs[regr_a] - encoder.num_tents],
                bweight_diff_ctrl[:, encoder.dm_idxs[regr_b] - encoder.num_tents],
            ).statistic
            for regr_b in encoder.dm_idxs.keys()
            if "tent" not in regr_b
        ]
        for regr_a in encoder.dm_idxs.keys()
        if "tent" not in regr_a
    ]
)
diff_corr_ctrl_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff_ctrl[
                        encoder.reg_idxs[reg],
                        encoder.dm_idxs[regr_a] - encoder.num_tents,
                    ],
                    bweight_diff_ctrl[
                        encoder.reg_idxs[reg],
                        encoder.dm_idxs[regr_b] - encoder.num_tents,
                    ],
                ).statistic
                for regr_b in encoder.dm_idxs.keys()
                if "tent" not in regr_b
            ]
            for regr_a in encoder.dm_idxs.keys()
            if "tent" not in regr_a
        ]
    )
    for reg in encoder.regions
}

plot_corr(diff_corr_ctrl)
plot_corr(diff_corr_ctrl_reg, reg="DLS")
plot_corr(diff_corr_ctrl_reg, reg="DMS")

In [ ]:
diff_corr_emc = diff_corr - diff_corr_ctrl
diff_corr_emc_reg = {
    reg: diff_corr_reg[reg] - diff_corr_ctrl_reg[reg] for reg in encoder.regions
}

plot_corr(diff_corr_emc)
plot_corr(diff_corr_emc_reg, reg="DLS")
plot_corr(diff_corr_emc_reg, reg="DMS")

## manifolds

In [ ]:
# compress task variable predicted activity for mb/mf to a principle component (pca or ae) and plot against each other

encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

robs_mb = encoder_mb.encoder.predict(encoder.tvs)
robs_mf = encoder_mf.encoder.predict(encoder.tvs)

In [ ]:
from sklearn.decomposition import PCA

pca_mb = PCA(n_components=1).fit(robs_mb)
pca_mf = PCA(n_components=1).fit(robs_mf)

pc_mb = pca_mb.transform(robs_mb)[:, 0]
pc_mf = pca_mf.transform(robs_mf)[:, 0]

pc_mb_trajs = pc_mb.reshape(encoder.num_trials, encoder.num_bins)
pc_mf_trajs = pc_mf.reshape(encoder.num_trials, encoder.num_bins)

In [ ]:
pca_mb.explained_variance_ratio_, pca_mf.explained_variance_ratio_

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
import numpy as np

fig, ax = plt.subplots()

im = ax.imshow(
    encoder_mb.encoder_weights[np.argsort(pca_mb.components_[0])]
    - encoder_mf.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
fig.colorbar(im)

In [ ]:
encoder_mb.dm_idxs

In [ ]:
import numpy as np

fig, axes = plt.subplots(
    nrows=1, ncols=2, sharey=True, figsize=(3, 2), constrained_layout=True
)

im = axes[0].imshow(
    encoder_mb.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
axes[1].imshow(
    encoder_mf.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)

axes[0].set_xlabel("regressor")

axes[0].set_ylabel("units (s.b. loadings)")
axes[1].set_ylabel("units (s.b. loadings)")

axes[0].set_title("mb")
axes[1].set_title("mf")

fig.colorbar(im, ax=axes, shrink=0.8, pad=0.05, label="encoder weight")

plt.show()

In [ ]:
from core.viz import plot_trajectory
import numpy as np

ax = plot_scatter(
    x=pc_mb,
    y=pc_mf,
    xlabel="mb",
    ylabel="mf",
    cmap="plasma",
    color=np.tile(range(encoder.num_bins), encoder.num_trials),
)

for i in range(encoder.num_trials):
    # s, e = i*encoder.num_bins, (i+1)*encoder.num_bins
    # plot_trajectory(x=pc_mb[s:e], y=pc_mf[s:e], ax=ax)
    plot_trajectory(x=pc_mb_trajs[i], y=pc_mf_trajs[i], ax=ax)

ax.scatter(
    x=pca_mb.components_[0],
    y=np.repeat(-6, encoder.num_units),
    cmap="plasma",
    c=np.argsort(pca_mb.components_[0]),
    alpha=0.5,
    s=0.5,
    marker="|",
)
ax.scatter(
    x=np.repeat(-7, encoder.num_units),
    y=pca_mf.components_[0],
    cmap="plasma",
    c=np.argsort(pca_mf.components_[0]),
    alpha=0.5,
    s=0.5,
    marker="|",
)

In [ ]:
trajs = np.array([pc_mb_trajs, pc_mf_trajs]).transpose(1, 0, 2)
n_unique = np.unique(trajs, axis=0).shape[0]
n_unique_mb = np.unique(pc_mb_trajs, axis=0).shape[0]
n_unique_mf = np.unique(pc_mf_trajs, axis=0).shape[0]

print("coords:", n_unique, "\nmb:", n_unique_mb, "\nmf:", n_unique_mf)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_clusters = range(2, trajs.shape[0] // 2)
seeds = range(10)
silhouettes = np.zeros((len(n_clusters), len(seeds)))

trajs_flat = trajs.reshape(trajs.shape[0], -1)

for i, n in enumerate(n_clusters):
    for j, seed in enumerate(seeds):
        km = KMeans(n_clusters=n, random_state=seed)
        cluster_labels = km.fit_predict(trajs_flat)
        silhouettes[i][j] = silhouette_score(trajs_flat, cluster_labels)

n_cluster = n_clusters[np.argmax(silhouettes.mean(axis=1))]
print("cluster w/ best avg. silhouette score: ", n)

In [ ]:
s_mean = silhouettes.mean(axis=1)
s_std = silhouettes.std(axis=1)
plt.figure(figsize=(2, 1), tight_layout=True)
plt.plot(n_clusters, s_mean)
plt.fill_between(n_clusters, s_mean - s_std, s_mean + s_std, alpha=0.5)
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder as OHE

trial_data_ohe = OHE().fit_transform(encoder.trial_data[encoder.tv_keys]).todense()

np.unique(trial_data_ohe, axis=0).shape

In [ ]:
km = KMeans(n_clusters=n_cluster, random_state=0)
cluster_labels = km.fit_predict(trajs_flat)

fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(6, 4), tight_layout=True)

for i, ax in enumerate(axes.flat):
    idxs = np.where(cluster_labels == i)[0]
    for j in idxs:
        ax = plot_trajectory(x=pc_mb_trajs[j], y=pc_mf_trajs[j], ax=ax)

    ax.set_xlim([-6, 6])
    ax.set_ylim([-6, 6])

    total = np.shape(idxs)[0]
    unique = np.unique(trajs[idxs], axis=0).shape[0]

    ax.set_title(f"cluster {i}, ({total}, {unique})")

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
# i would wnat to still look at communication subspace and see the difference in task variables encoding in the private and shared subspace

In [ ]:
np.where(cluster_labels == i)
# are neurons that evlve along the axis for one task variable the same neurons that do something wacky for other task variables?
# plot all the trajectories in dms and dls on the same figure and

## the t-population

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy